# YZM212 Makine Öğrenmesi - 4. Ödev

**İsim-Soyisim:** Görkem Özer  
**Numara:** 23291007

## Problem Tanımı
Gürültülü gözlem verilerinden Bayesyen yöntem ile μ ve σ tahmini.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
import corner

true_mu = 150.0
true_sigma = 10.0
n_obs = 50

np.random.seed(42)
data = true_mu + true_sigma * np.random.randn(n_obs)

def log_likelihood(theta, data):
    mu, sigma = theta
    if sigma <= 0:
        return -np.inf
    return -0.5 * np.sum(((data - mu) / sigma)**2 + np.log(2 * np.pi * sigma**2))

def log_prior(theta):
    mu, sigma = theta
    if 0 < mu < 300 and 0 < sigma < 50:
        return 0.0
    return -np.inf

def log_probability(theta, data):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, data)

initial = [140, 5]
n_walkers = 32
pos = initial + 1e-4 * np.random.randn(n_walkers, 2)

sampler = emcee.EnsembleSampler(n_walkers, 2, log_probability, args=(data,))
sampler.run_mcmc(pos, 2000, progress=True)

flat_samples = sampler.get_chain(discard=500, thin=15, flat=True)

fig = corner.corner(flat_samples, labels=["mu", "sigma"], truths=[true_mu, true_sigma])
plt.show()

mu_median = np.median(flat_samples[:, 0])
sigma_median = np.median(flat_samples[:, 1])

mu_low, mu_high = np.percentile(flat_samples[:, 0], [16, 84])
sigma_low, sigma_high = np.percentile(flat_samples[:, 1], [16, 84])

print("Mu:", mu_median, mu_low, mu_high)
print("Sigma:", sigma_median, sigma_low, sigma_high)
